# 07 · `dinov2_frozen` — benchmark común + ablación (Integrante 2)

**Arquitectura:** DINOv2 ViT-S/14 **100 % congelado** (0 parámetros entrenables en el backbone) + *neck* piramidal + `UNetDecoder` común. Es el *linear probing* con decodificador: mide qué tan buenas son las features auto-supervisadas **tal cual**.

**Tu pregunta de ablación:** como el backbone no aprende, todo depende del *neck*. ¿Agregar una conv 3×3 tras el reescalado bilineal (suaviza la cuadrícula de 14 px) mejora los bordes? ¿El modelo estaba sub-entrenado (más LR del decoder / más épocas)?

### Protocolo de 2 niveles (por qué hay celdas que no se tocan)
| Nivel | Qué es | Qué cambia |
|---|---|---|
| **1 · Benchmark principal** | corrida `base` de los 3 modelos | **nada**: mismos parches 512 × 512, mismo *data augmentation*, mismo `UNetDecoder` (3,098,433 parámetros), misma pérdida y métricas |
| **2 · Ablación** | tus corridas extra en `RUNS` | **solo** los parámetros propios de tu *backbone* listados en `ALLOWED` (una hipótesis → un cambio → una medición) |

El notebook se protege solo: si cambias algo común, o un parámetro que no es de tu modelo, se detiene con un mensaje que explica qué pasó.

### Qué puedes tocar (solo en la celda 2 · `RUNS`)
| Parámetro | Base | Ideas para probar | Qué hace |
|---|---|---|---|
| `NECK_TYPE` | `"bilinear_1x1"` | `"bilinear_3x3"`, `"custom"` | diseño del neck (sección 7.2 para el tuyo) |
| `NECK_REFINE_SCALES` | `(4, 8)` | `(4,)`, `(4, 8, 16)` | en qué escalas `bilinear_3x3` agrega la conv 3×3 |
| `LR_DECODER` | `3e-4` | `6e-4`, `1e-3` | LR del neck + decoder (lo único que aprende) |
| `EPOCHS` | `20` | `30` | la curva de validación seguía subiendo en la época 20 |
| `DINO_LAYER_IDS` | `(2, 5, 8, 11)` | `(3, 6, 9, 11)`, `(5, 7, 9, 11)` | de qué bloques del ViT salen los 4 mapas del U-Net |

El **umbral calibrado en validación** y el **TTA** ya se reportan automáticamente para todos los modelos (columnas extra de la sección 9).

### Cómo ejecutar
1. **Kaggle** (recomendado): *Accelerator* GPU T4, *Internet* On, *Add Input* → `balraj98/massachusetts-roads-dataset`. Luego *Run All*.
2. **Colab**: GPU T4; el dataset se descarga solo con `kagglehub` (si falla, súbelo a `/content`).
3. Primero prueba con `SMOKE_TEST = True` (≈ 5 min) y luego ponlo en `False` para las corridas reales.
4. Al final envía `outputs_{CONDITION}.zip` (o los `.json` de `results/`) al grupo.

## 0. Dependencias
Kaggle y Colab ya traen `torch`, `torchvision`, `pandas`, `matplotlib` y `tqdm`. DINOv2 se descarga con `torch.hub` (necesita **Internet On**).

In [ ]:
import importlib.util
import subprocess
import sys

for pkg, mod in [("pandas", "pandas"), ("tqdm", "tqdm"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Dependencias listas.")

## 1. 🔒 Configuración común del grupo (NO EDITAR)
Esta celda es **idéntica en los 3 notebooks**. Define todo lo que hace justa la comparación (Nivel 1 del protocolo):

- **Datos:** mismo re-split por tile (`SPLIT_SEED = 42`, 70/15/15 → 820/176/175 tiles) y misma rejilla fija de parches de 512 × 512.
- **Data augmentation:** la misma función y los mismos parámetros (`AUG`) para las 3 arquitecturas.
- **Decodificador:** `UNetDecoder` con `DECODER_CH = (64, 128, 256, 512)` → exactamente **3,098,433** parámetros.
- **Pérdida, métricas y evaluación:** BCE + Dice, `MetricAccumulator` global, umbral oficial 0.5, ρ = 3 px.
- **`MODEL_BASE`:** los valores de la fila *base* de cada modelo. **`ALLOWED`:** qué parámetros puede cambiar cada integrante en su ablación.

Al final se calcula una **huella** (*fingerprint*) de esta configuración. Si alguien la modifica sin querer, el notebook se detiene con un error. Si el grupo decide cambiar algo común (p. ej. el *augmentation*), se cambia **en los 3 notebooks a la vez** y se actualiza `EXPECTED_FINGERPRINT`.

In [ ]:
import json, hashlib

COMMON = dict(
    # --- datos ---
    SEED=42,
    SPLIT_SEED=42,
    SPLIT_FRACS=(0.70, 0.15, 0.15),     # train / val / test, a nivel de TILE (evita fuga espacial)
    PATCH=512,
    GRID_STARTS=(0, 494, 988),          # rejilla 3x3 de parches de 512 sobre tiles de 1500 px
    BLANK_THR=0.25,                     # se descartan parches con > 25 % de píxeles blancos (sin datos)
    # --- data augmentation (IGUAL para los 3 modelos, solo en train) ---
    AUG=dict(
        HFLIP_P=0.5, VFLIP_P=0.5, ROT90=True,                  # geométricas (imagen y máscara)
        BRIGHTNESS=0.15, CONTRAST=0.15, SATURATION=0.15,       # color (solo imagen)
        SHADOW_P=0.3, SHADOW_MAX=3,                            # sombras sintéticas (oclusión de árboles/edificios)
        SHADOW_STRENGTH=(0.40, 0.75), SHADOW_SIZE=(0.05, 0.25),
    ),
    # --- arquitectura común ---
    DECODER_CH=(64, 128, 256, 512),
    # --- entrenamiento ---
    EPOCHS=20,
    BATCH_SIZE=8,
    WARMUP_FRAC=0.05,
    GRAD_CLIP=1.0,
    AMP=True,
    # --- pérdida ---
    LOSS_TYPE="bce_dice",
    BCE_POS_WEIGHT=1.0,
    FOCAL_GAMMA=2.0,
    # --- evaluación ---
    THRESHOLD=0.5,                      # umbral OFICIAL (tabla principal)
    RELAX_RHO=3,                        # tolerancia (px) del F1 relajado
    VAL_THRESHOLDS=(0.35, 0.40, 0.45, 0.50, 0.55, 0.60),   # barrido extra en validación
)

# Valores de la fila "base" de cada arquitectura (los de los notebooks 03-05).
MODEL_BASE = {
    "resnet50": dict(RESNET_WEIGHTS="imagenet", ADAPTER_TYPE="conv1x1", BOTTLENECK_DROPOUT=0.0, FREEZE_STAGES=0,
                     LR_DECODER=3e-4, LR_BACKBONE=1e-4, WEIGHT_DECAY=1e-4),
    "dinov2_frozen": dict(DINO_VARIANT="dinov2_vits14", DINO_INPUT=518, DINO_LAYER_IDS=(2, 5, 8, 11),
                          NECK_TYPE="bilinear_1x1", NECK_REFINE_SCALES=(4, 8),
                          N_UNFREEZE=0, UNFREEZE_BLOCKS=None, LR_DINO_TAIL=0.0, LLRD_DECAY=1.0,
                          LR_DECODER=3e-4, WEIGHT_DECAY=1e-4),
    "dinov2_partial": dict(DINO_VARIANT="dinov2_vits14", DINO_INPUT=518, DINO_LAYER_IDS=(2, 5, 8, 11),
                           NECK_TYPE="bilinear_1x1", NECK_REFINE_SCALES=(4, 8),
                           N_UNFREEZE=2, UNFREEZE_BLOCKS=None, LR_DINO_TAIL=1e-5, LLRD_DECAY=1.0,
                           LR_DECODER=3e-4, WEIGHT_DECAY=1e-4),
}

# Lo que cada integrante PUEDE cambiar en sus corridas de ablación (Nivel 2).
ALLOWED = {
    "resnet50": ["WEIGHT_DECAY", "LR_BACKBONE", "LR_DECODER", "BOTTLENECK_DROPOUT", "FREEZE_STAGES", "ADAPTER_TYPE"],
    "dinov2_frozen": ["NECK_TYPE", "NECK_REFINE_SCALES", "LR_DECODER", "WEIGHT_DECAY", "EPOCHS", "DINO_LAYER_IDS"],
    "dinov2_partial": ["N_UNFREEZE", "UNFREEZE_BLOCKS", "LR_DINO_TAIL", "LLRD_DECAY", "LR_DECODER", "WEIGHT_DECAY",
                       "LOSS_TYPE", "BCE_POS_WEIGHT", "FOCAL_GAMMA"],
}

EXPECTED_DECODER_PARAMS = 3_098_433


def config_fingerprint(*objs):
    """Huella corta (sha1) de la configuración común: igual en los 3 notebooks si nadie la tocó."""
    return hashlib.sha1(json.dumps(objs, sort_keys=True, default=str).encode()).hexdigest()[:12]


EXPECTED_FINGERPRINT = "234ea433fba9"
LOCKED_FINGERPRINT = config_fingerprint(COMMON, MODEL_BASE, ALLOWED)
assert LOCKED_FINGERPRINT == EXPECTED_FINGERPRINT, (
    f"La configuración común cambió (huella {LOCKED_FINGERPRINT} != {EXPECTED_FINGERPRINT}). "
    "Deshaz el cambio o acuérdalo con el grupo y aplícalo en los 3 notebooks.")
print("Configuración común OK · huella", LOCKED_FINGERPRINT)

## 2. ✏️ Tu zona de configuración
**Es la única celda que necesitas editar.** Cada entrada de `RUNS` es una corrida completa; `"base"` queda vacía. Si pones un parámetro que no es de tu modelo, el notebook te avisa con la lista de los permitidos.

In [ ]:
# =====================================================================================
# 2. ✏️ TU ZONA DE CONFIGURACIÓN — lo único que necesitas editar
# =====================================================================================
CONDITION = "dinov2_frozen"   # NO cambiar: este notebook es el de DINOv2 congelado

# Cada entrada de RUNS es UNA corrida de entrenamiento completa.
#   - "base" debe quedar vacía: es tu fila del benchmark común (Nivel 1).
#   - Las demás son tus ablaciones (Nivel 2): cambia UNA cosa por corrida.
#   - Solo se aceptan claves de ALLOWED["dinov2_frozen"]:
#       NECK_TYPE, NECK_REFINE_SCALES, LR_DECODER, WEIGHT_DECAY, EPOCHS, DINO_LAYER_IDS
RUNS = {
    "base": {},
    "neck_3x3": {"NECK_TYPE": "bilinear_3x3"},                  # ablación asignada en el plan
    # "neck_3x3_all": {"NECK_TYPE": "bilinear_3x3", "NECK_REFINE_SCALES": (4, 8, 16, 32)},
    # "lr_dec_6e-4": {"LR_DECODER": 6e-4},                      # la cabeza aprende más rápido
    # "epochs_30": {"EPOCHS": 30},                              # ¿estaba sub-entrenado? (reportar el costo extra)
    # "layers_3_6_9_11": {"DINO_LAYER_IDS": (3, 6, 9, 11)},     # features un poco más profundas en los skips
    # "custom_neck": {"NECK_TYPE": "custom"},                   # tu neck de la sección 7.2
}

SMOKE_TEST = False     # True = 1 época corta por corrida, solo para verificar que todo corre
SKIP_IF_DONE = True    # si ya existe el JSON de una corrida, no se vuelve a entrenar

### Preparación del entorno
Imports, rutas de salida (`/kaggle/working` en Kaggle, `/content/outputs` en Colab) y semillas.

In [ ]:
import os, sys, json, math, time, glob, random, copy, contextlib, shutil, hashlib
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor
os.environ.setdefault("XFORMERS_DISABLED", "1")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ON_KAGGLE = os.path.isdir("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else ("/content/outputs" if os.path.isdir("/content") else os.path.abspath("outputs"))
RESULTS_DIR, FIG_DIR, CKPT_DIR = f"{WORK}/results", f"{WORK}/figures", f"{WORK}/checkpoints"
PREP_DIR = "/tmp/massachusetts_512"      # dataset de parches fijos (disco local, no ocupa el output de Kaggle)
for d in [RESULTS_DIR, FIG_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
assert CONDITION in MODEL_BASE, CONDITION
for name, ov in RUNS.items():
    if name == "base":
        assert ov == {}, "La corrida 'base' debe quedar vacía: es la fila de control del grupo."


def seed_everything(seed):
    """Fija todas las semillas relevantes para reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_fig(name):
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=130, bbox_inches="tight")
    plt.show()


def to_jsonable(o):
    if isinstance(o, SimpleNamespace):
        return to_jsonable(vars(o))
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if math.isnan(float(o)) else float(o)
    return o


def denorm(x):
    """Tensor normalizado (3,H,W) -> imagen float [0,1] (H,W,3)."""
    return (x.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()


seed_everything(COMMON["SEED"])
print("torch", torch.__version__, "| device:", DEVICE, "| salida en", WORK)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

## 3. Dataset fijo `massachusetts_512` (idéntico para los 3 modelos)
En lugar de recortar al vuelo (notebooks 03–05), aquí se **pre-generan una sola vez** todos los parches de 512 × 512:

1. Se juntan los 1171 tiles originales (1500 × 1500) y se re-parten **por tile** con `SPLIT_SEED = 42` → 820 / 176 / 175.
2. Cada tile se corta en la rejilla fija 3 × 3 (inicios 0, 494, 988) y se descartan los parches con más de 25 % de blanco.
3. Se guardan como PNG en `/tmp/massachusetts_512/` (`images/`, `masks/`, `patches.csv`).

Ventajas: la época 1 ya no tarda 10–25 min (se leía el TIFF pesado desde la red), y **los 3 modelos ven exactamente los mismos parches**, sin la suerte de los recortes aleatorios. Al final se imprime la **huella del dataset**: compárala con tus compañeros, debe ser la misma (también se guarda en cada JSON de resultados).

Si ya generaste el dataset y lo subiste como Kaggle Dataset (carpeta con `massachusetts_512_meta.json`), se usa ese directamente.

In [ ]:
def find_raw_massachusetts():
    """Raíz con train/val/test y *_labels (Kaggle input o descarga con kagglehub en Colab)."""
    roots = ["/kaggle/input", "/content"]
    for base in roots:
        for c in glob.glob(f"{base}/**/train_labels", recursive=True):
            root = os.path.dirname(c)
            if all(os.path.isdir(f"{root}/{d}") for d in ["train", "val", "test", "train_labels", "val_labels", "test_labels"]):
                return root
    try:
        import kagglehub
        path = kagglehub.dataset_download("balraj98/massachusetts-roads-dataset")
        for c in glob.glob(f"{path}/**/train_labels", recursive=True):
            return os.path.dirname(c)
    except Exception as e:
        print("kagglehub no disponible:", e)
    raise FileNotFoundError("No se encontró Massachusetts Roads. En Kaggle: Add Input -> balraj98/massachusetts-roads-dataset.")


def list_pairs(root, split):
    exts = (".tif", ".tiff", ".png")
    imgs = {os.path.splitext(f)[0]: f"{root}/{split}/{f}" for f in os.listdir(f"{root}/{split}") if f.lower().endswith(exts)}
    lbls = {os.path.splitext(f)[0]: f"{root}/{split}_labels/{f}" for f in os.listdir(f"{root}/{split}_labels") if f.lower().endswith(exts)}
    return [(k, imgs[k], lbls[k]) for k in sorted(set(imgs) & set(lbls))]


def resplit_tiles(root):
    """Junta train+val+test originales y re-parte POR TILE (determinista, igual que en los notebooks 03-05)."""
    pool = sorted(list_pairs(root, "train") + list_pairs(root, "val") + list_pairs(root, "test"), key=lambda p: p[0])
    assert len({p[0] for p in pool}) == len(pool), "Nombres de tile duplicados entre splits"
    perm = np.random.RandomState(COMMON["SPLIT_SEED"]).permutation(len(pool))
    n_tr = int(round(COMMON["SPLIT_FRACS"][0] * len(pool)))
    n_va = int(round(COMMON["SPLIT_FRACS"][1] * len(pool)))
    idx = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}
    return {s: [pool[i] for i in sorted(ix)] for s, ix in idx.items()}


def blank_fraction(rgb):
    return float((rgb.min(axis=2) >= 250).mean())


def cut_tile(args):
    """Corta un tile en la rejilla fija y guarda los parches válidos. Devuelve filas para patches.csv."""
    split, name, ip, lp, out = args
    P, rows = COMMON["PATCH"], []
    rgb = np.array(Image.open(ip).convert("RGB"))
    lab = (np.array(Image.open(lp).convert("L")) > 127).astype(np.uint8)
    for oy in COMMON["GRID_STARTS"]:
        for ox in COMMON["GRID_STARTS"]:
            img, m = rgb[oy:oy + P, ox:ox + P], lab[oy:oy + P, ox:ox + P]
            if img.shape[:2] != (P, P) or blank_fraction(img) > COMMON["BLANK_THR"]:
                continue
            pid = f"{name}_{oy:04d}_{ox:04d}"
            Image.fromarray(img).save(f"{out}/images/{pid}.png", compress_level=1)
            Image.fromarray(m * 255).save(f"{out}/masks/{pid}.png", compress_level=1)
            rows.append({"id": pid, "tile": name, "split": split, "oy": oy, "ox": ox, "road_frac": round(float(m.mean()), 6)})
    return rows


def dataset_fingerprint(df):
    s = "\n".join(f"{r.id},{r.split},{r.road_frac:.6f}" for r in df.sort_values("id").itertuples())
    return hashlib.sha1(s.encode()).hexdigest()[:12]


def prepare_massachusetts_512(out=PREP_DIR):
    for meta in glob.glob("/kaggle/input/**/massachusetts_512_meta.json", recursive=True) + [f"{out}/massachusetts_512_meta.json"]:
        if os.path.exists(meta):
            root = os.path.dirname(meta)
            print("Dataset de parches ya existe:", root)
            return root
    raw = find_raw_massachusetts()
    splits = resplit_tiles(raw)
    os.makedirs(f"{out}/images", exist_ok=True)
    os.makedirs(f"{out}/masks", exist_ok=True)
    jobs = [(s, n, ip, lp, out) for s in ["train", "val", "test"] for (n, ip, lp) in splits[s]]
    rows = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        for r in tqdm(ex.map(cut_tile, jobs), total=len(jobs), desc="cortando tiles"):
            rows += r
    df = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
    df.to_csv(f"{out}/patches.csv", index=False)
    meta = {"tiles": {s: len(v) for s, v in splits.items()}, "patches": df.split.value_counts().to_dict(),
            "fingerprint": dataset_fingerprint(df), "common": {k: COMMON[k] for k in ["SPLIT_SEED", "SPLIT_FRACS", "PATCH", "GRID_STARTS", "BLANK_THR"]}}
    with open(f"{out}/massachusetts_512_meta.json", "w") as f:
        json.dump(to_jsonable(meta), f, indent=2)
    return out


DATA_ROOT = prepare_massachusetts_512()
PATCHES = pd.read_csv(f"{DATA_ROOT}/patches.csv")
DATASET_FP = dataset_fingerprint(PATCHES)
print("Parches por split:", PATCHES.split.value_counts().to_dict())
print("Fracción media de vía por split:", PATCHES.groupby("split").road_frac.mean().round(4).to_dict())
print("HUELLA DEL DATASET:", DATASET_FP, "<- debe coincidir en los 3 notebooks")
# Referencia de los notebooks 03-05 (misma rejilla en val/test): val = 1363, test = 1339 parches.
for s, ref in [("val", 1363), ("test", 1339)]:
    n = int((PATCHES.split == s).sum())
    if n != ref:
        print(f"AVISO: {s} tiene {n} parches (referencia 03-05: {ref}). Revisa la versión del dataset de entrada.")

## 4. Data augmentation común (solo en train)
`augment()` es la **misma función con los mismos parámetros (`COMMON["AUG"]`) para las 3 arquitecturas**. Si un modelo tuviera más augmentation que otro, no sabríamos si gana por el *backbone* o por los datos (D09 de `DECISIONS.md`).

| Paso | Qué hace | Afecta a |
|---|---|---|
| Volteo horizontal / vertical (p = 0.5) | espejo | imagen **y** máscara |
| Rotación 90° × k (k aleatorio) | las vías no tienen orientación preferida | imagen **y** máscara |
| Brillo / contraste / saturación ±15 % | cambios de iluminación y sensor | solo imagen |
| **Sombras sintéticas** (p = 0.3, hasta 3 elipses suaves que oscurecen 25–60 %) | simula árboles o edificios tapando la vía | solo imagen: la máscara **no** cambia, así el modelo aprende a seguir la vía aunque esté oscura |

La celda siguiente muestra ejemplos: úsala para entender qué ve el modelo durante el entrenamiento.

In [ ]:
def add_shadows(x, cfg):
    """Oscurece hasta SHADOW_MAX elipses suaves. x float (H,W,3) en [0,1]. La máscara no se toca."""
    H, W = x.shape[:2]
    yy, xx = np.ogrid[:H, :W]
    for _ in range(np.random.randint(1, cfg["SHADOW_MAX"] + 1)):
        cy, cx = np.random.uniform(0, H), np.random.uniform(0, W)
        ry, rx = (np.random.uniform(*cfg["SHADOW_SIZE"], size=2) * np.array([H, W]))
        d = np.sqrt(((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2)
        alpha = np.clip((1.0 - d) / 0.3, 0.0, 1.0)[..., None]          # borde difuso
        factor = np.random.uniform(*cfg["SHADOW_STRENGTH"])            # 0.4 = sombra fuerte, 0.75 = suave
        x = x * (1.0 - alpha * (1.0 - factor))
    return x


def augment(img, mask, cfg):
    """img uint8 (H,W,3), mask uint8 (H,W) -> img float [0,1], mask uint8. cfg = COMMON["AUG"]."""
    if np.random.rand() < cfg["HFLIP_P"]:
        img, mask = img[:, ::-1], mask[:, ::-1]
    if np.random.rand() < cfg["VFLIP_P"]:
        img, mask = img[::-1], mask[::-1]
    if cfg["ROT90"]:
        k = np.random.randint(4)
        img, mask = np.rot90(img, k), np.rot90(mask, k)
    x = img.astype(np.float32) / 255.0
    x = x * (1 + np.random.uniform(-cfg["BRIGHTNESS"], cfg["BRIGHTNESS"]))          # brillo
    m = x.mean()
    x = (x - m) * (1 + np.random.uniform(-cfg["CONTRAST"], cfg["CONTRAST"])) + m    # contraste
    g = x.mean(axis=2, keepdims=True)
    x = (x - g) * (1 + np.random.uniform(-cfg["SATURATION"], cfg["SATURATION"])) + g  # saturación
    if np.random.rand() < cfg["SHADOW_P"]:
        x = add_shadows(x, cfg)
    return np.clip(x, 0, 1), np.ascontiguousarray(mask)


def to_tensors(img, mask):
    """img uint8 o float[0,1] (H,W,3); mask (H,W) -> tensores normalizados (3,H,W) y (1,H,W)."""
    x = img.astype(np.float32) / 255.0 if img.dtype == np.uint8 else img.astype(np.float32)
    x = (torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1) - MEAN) / STD
    return x, torch.from_numpy(np.ascontiguousarray(mask)).float().unsqueeze(0)


class Patches512(Dataset):
    """Lee los parches fijos de patches.csv. Augmentation solo si split == 'train'."""

    def __init__(self, root, split, aug_cfg):
        self.root, self.train, self.aug_cfg = root, split == "train", aug_cfg
        self.ids = PATCHES.loc[PATCHES.split == split, "id"].tolist()
        if not self.ids:
            raise ValueError(f"Sin parches en el split '{split}'")

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, k):
        pid = self.ids[k]
        img = np.array(Image.open(f"{self.root}/images/{pid}.png").convert("RGB"))
        m = (np.array(Image.open(f"{self.root}/masks/{pid}.png").convert("L")) > 127).astype(np.uint8)
        if self.train:
            img, m = augment(img, m, self.aug_cfg)
        return to_tensors(img, m)


def worker_init(worker_id):
    s = (torch.initial_seed() + worker_id) % 2 ** 32
    np.random.seed(s)
    random.seed(s)


def make_loader(ds, cfg, shuffle, drop_last=False):
    return DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=shuffle, drop_last=drop_last, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda", worker_init_fn=worker_init,
                      generator=torch.Generator().manual_seed(cfg.SEED))

In [ ]:
# Vista previa del augmentation: misma imagen, 4 sorteos distintos (fila de arriba imagen, abajo máscara).
np.random.seed(0)
_ids = PATCHES[(PATCHES.split == "train") & (PATCHES.road_frac > 0.05)].id.tolist()
_img = np.array(Image.open(f"{DATA_ROOT}/images/{_ids[0]}.png").convert("RGB"))
_msk = (np.array(Image.open(f"{DATA_ROOT}/masks/{_ids[0]}.png").convert("L")) > 127).astype(np.uint8)
fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
axes[0, 0].imshow(_img); axes[0, 0].set_title("original"); axes[1, 0].imshow(_msk, cmap="gray")
for j in range(1, 5):
    a, b = augment(_img, _msk, COMMON["AUG"])
    axes[0, j].imshow(a); axes[0, j].set_title(f"aumento {j}"); axes[1, j].imshow(b, cmap="gray")
for ax in axes.ravel():
    ax.axis("off")
save_fig("augmentation_preview")

## 5. Modelo = encoder → adaptador/neck → `UNetDecoder` común
```
imagen 512×512 ──► ENCODER (ResNet-50 ó DINOv2) ──► 4 mapas ──► ADAPTADOR / NECK ──► 4 mapas con canales (64,128,256,512)
                                                                                    en strides /4, /8, /16, /32
                                                                                             │
                                                         máscara 512×512 (1 logit) ◄── UNetDecoder (3.10 M, IGUAL para todos)
```
- **`ResNet50Encoder`**: devuelve `layer1..layer4` (256/512/1024/2048 canales en /4, /8, /16, /32). Su **adaptador** (`ResNetAdapter`) es una conv 1×1 por nivel.
- **`DinoV2Encoder`**: ViT-S/14, la imagen se reescala a 518 = 37 × 14 → rejilla de 37 × 37 tokens de 384 dims. Toma la salida de 4 bloques (`DINO_LAYER_IDS`, base `[2, 5, 8, 11]`). Como un ViT no tiene pirámide, el **neck** (`NECKS[...]`) reescala cada mapa a 128/64/32/16 px.
- **`UNetDecoder`**: sube de /32 a /4 con *skip connections* y 2 convs 3×3 por nivel, y luego ×4 hasta 512. **No se toca.**

Los componentes intercambiables están en *registros* (diccionarios): `ADAPTERS`, `NECKS`, `LOSSES`. Elegir uno es cambiar un string en `RUNS` (p. ej. `"NECK_TYPE": "bilinear_3x3"`). Si quieres programar tu propia variante, la agregas al registro en tu **zona de código libre** (sección 7.2).

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, cin, cout, k=3):
        super().__init__(nn.Conv2d(cin, cout, k, padding=k // 2, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


# ============================ ResNet-50 ============================
class ResNet50Encoder(nn.Module):
    """ResNet-50 de torchvision -> [layer1 (/4), layer2 (/8), layer3 (/16), layer4 (/32)].
    freeze_stages = k congela las k primeras etapas de [stem, layer1, layer2, layer3, layer4]."""
    out_channels = (256, 512, 1024, 2048)

    def __init__(self, weights="imagenet", freeze_stages=0):
        super().__init__()
        w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if weights == "imagenet" else None
        m = torchvision.models.resnet50(weights=w)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = m.layer1, m.layer2, m.layer3, m.layer4
        self.frozen = [self.stem, self.layer1, self.layer2, self.layer3, self.layer4][:freeze_stages]
        for mod in self.frozen:
            for p in mod.parameters():
                p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        for mod in self.frozen:          # las etapas congeladas mantienen sus estadísticas de BatchNorm
            mod.eval()
        return self

    def forward(self, x):
        f1 = self.layer1(self.stem(x))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [f1, f2, f3, f4]


class ResNetAdapter(nn.Module):
    """Base: conv 1x1 + BN + ReLU por nivel (canales ResNet -> DECODER_CH). Opcional: Dropout2d en /32."""

    def __init__(self, in_chs, out_chs, cfg):
        super().__init__()
        self.proj = nn.ModuleList([ConvBNReLU(i, o, k=1) for i, o in zip(in_chs, out_chs)])
        self.drop = nn.Dropout2d(cfg.BOTTLENECK_DROPOUT) if cfg.BOTTLENECK_DROPOUT > 0 else nn.Identity()

    def forward(self, feats, size):
        outs = [p(f) for p, f in zip(self.proj, feats)]
        outs[-1] = self.drop(outs[-1])
        return outs


ADAPTERS = {"conv1x1": ResNetAdapter}


# ============================ DINOv2 ============================
def dino_trainable_blocks(cfg, n_blocks):
    """Bloques a descongelar: UNFREEZE_BLOCKS explícito, o los últimos N_UNFREEZE."""
    if cfg.UNFREEZE_BLOCKS is not None:
        return sorted(int(b) for b in cfg.UNFREEZE_BLOCKS)
    return list(range(n_blocks - cfg.N_UNFREEZE, n_blocks)) if cfg.N_UNFREEZE > 0 else []


class DinoV2Encoder(nn.Module):
    """DINOv2 de torch.hub. Devuelve 4 mapas (B, 384, 37, 37) de los bloques layer_ids."""

    def __init__(self, cfg):
        super().__init__()
        self.input_size, self.layer_ids = cfg.DINO_INPUT, list(cfg.DINO_LAYER_IDS)
        self.backbone = torch.hub.load("facebookresearch/dinov2", cfg.DINO_VARIANT, trust_repo=True)
        self.embed_dim, self.n_blocks = self.backbone.embed_dim, len(self.backbone.blocks)
        assert len(self.layer_ids) == 4 and self.layer_ids == sorted(self.layer_ids) and self.layer_ids[-1] < self.n_blocks, self.layer_ids
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.trainable_blocks = dino_trainable_blocks(cfg, self.n_blocks)
        for b in self.trainable_blocks:
            for p in self.backbone.blocks[b].parameters():
                p.requires_grad = True
        if self.trainable_blocks:                    # la LayerNorm final se adapta junto con la cola
            for p in self.backbone.norm.parameters():
                p.requires_grad = True

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()      # DINOv2 no tiene BN/dropout activo: siempre en eval
        return self

    def forward(self, x):
        x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        ctx = contextlib.nullcontext() if self.trainable_blocks else torch.no_grad()
        with ctx:
            feats = self.backbone.get_intermediate_layers(x, n=self.layer_ids, reshape=True, norm=True)
        return list(feats)


class NeckBilinear1x1(nn.Module):
    """BASE (estilo ViTDet simple): reescalado bilineal a /4, /8, /16, /32 + conv 1x1 + BN + ReLU."""
    strides = (4, 8, 16, 32)

    def __init__(self, embed_dim, out_chs, cfg):
        super().__init__()
        self.proj = nn.ModuleList([ConvBNReLU(embed_dim, o, k=1) for o in out_chs])

    def forward(self, feats, size):
        H, W = size
        return [p(F.interpolate(f, size=(H // s, W // s), mode="bilinear", align_corners=False))
                for f, p, s in zip(feats, self.proj, self.strides)]


class NeckBilinear3x3(NeckBilinear1x1):
    """Igual que la base + una conv 3x3 extra en las escalas NECK_REFINE_SCALES (suaviza la cuadrícula de 14 px)."""

    def __init__(self, embed_dim, out_chs, cfg):
        super().__init__(embed_dim, out_chs, cfg)
        self.refine = nn.ModuleList([ConvBNReLU(o, o, k=3) if s in cfg.NECK_REFINE_SCALES else nn.Identity()
                                     for o, s in zip(out_chs, self.strides)])

    def forward(self, feats, size):
        return [r(f) for r, f in zip(self.refine, super().forward(feats, size))]


NECKS = {"bilinear_1x1": NeckBilinear1x1, "bilinear_3x3": NeckBilinear3x3}


# ============================ Decodificador común (NO TOCAR) ============================
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(ConvBNReLU(cin, cout, 3), ConvBNReLU(cout, cout, 3))


class UNetDecoder(nn.Module):
    """De /32 a /4 (upsample x2 + skip + 2 convs) y luego x4 hasta la entrada -> 1 logit."""

    def __init__(self, chs):
        super().__init__()
        c4, c8, c16, c32 = chs
        self.up16 = DoubleConv(c32 + c16, c16)
        self.up8 = DoubleConv(c16 + c8, c8)
        self.up4 = DoubleConv(c8 + c4, c4)
        self.head = nn.Conv2d(c4, 1, kernel_size=1)

    def forward(self, feats, size):
        f4, f8, f16, f32 = feats
        x = self.up16(torch.cat([F.interpolate(f32, size=f16.shape[-2:], mode="bilinear", align_corners=False), f16], 1))
        x = self.up8(torch.cat([F.interpolate(x, size=f8.shape[-2:], mode="bilinear", align_corners=False), f8], 1))
        x = self.up4(torch.cat([F.interpolate(x, size=f4.shape[-2:], mode="bilinear", align_corners=False), f4], 1))
        x = F.interpolate(x, size=size, mode="bilinear", align_corners=False)
        return self.head(x)


class SegModel(nn.Module):
    def __init__(self, encoder, adapter, decoder):
        super().__init__()
        self.encoder, self.adapter, self.decoder = encoder, adapter, decoder

    def forward(self, x):
        size = tuple(x.shape[-2:])
        return self.decoder(self.adapter(self.encoder(x), size), size)


def build_model(cfg, condition):
    if condition == "resnet50":
        enc = ResNet50Encoder(cfg.RESNET_WEIGHTS, cfg.FREEZE_STAGES)
        adapter = ADAPTERS[cfg.ADAPTER_TYPE](enc.out_channels, cfg.DECODER_CH, cfg)
    else:
        enc = DinoV2Encoder(cfg)
        adapter = NECKS[cfg.NECK_TYPE](enc.embed_dim, cfg.DECODER_CH, cfg)
    return SegModel(enc, adapter, UNetDecoder(cfg.DECODER_CH))


def param_groups(model, cfg, condition):
    """AdamW con LR distinto: cabeza (adaptador/neck + decoder) vs. parte entrenable del encoder."""
    head = [p for m in (model.adapter, model.decoder) for p in m.parameters() if p.requires_grad]
    groups = [{"params": head, "lr": cfg.LR_DECODER, "weight_decay": cfg.WEIGHT_DECAY, "name": "head"}]
    enc = model.encoder
    if condition == "resnet50":
        ps = [p for p in enc.parameters() if p.requires_grad]
        if ps:
            groups.append({"params": ps, "lr": cfg.LR_BACKBONE, "weight_decay": cfg.WEIGHT_DECAY, "name": "backbone"})
    elif enc.trainable_blocks:
        last = max(enc.trainable_blocks)
        for b in enc.trainable_blocks:      # Layer-wise LR decay: bloques más tempranos -> LR más bajo
            groups.append({"params": list(enc.backbone.blocks[b].parameters()), "lr": cfg.LR_DINO_TAIL * cfg.LLRD_DECAY ** (last - b),
                           "weight_decay": cfg.WEIGHT_DECAY, "name": f"dino_block_{b}"})
        groups.append({"params": list(enc.backbone.norm.parameters()), "lr": cfg.LR_DINO_TAIL,
                       "weight_decay": cfg.WEIGHT_DECAY, "name": "dino_norm"})
    return groups


def n_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))


_dec = UNetDecoder(COMMON["DECODER_CH"])
assert n_params(_dec) == EXPECTED_DECODER_PARAMS, n_params(_dec)
print(f"Decodificador común: {n_params(_dec):,} parámetros (idéntico en los 3 notebooks).")
del _dec

## 6. Pérdida y métricas (comunes)
- **Pérdida base:** `bce_dice` = BCE + Dice suave (Dice compensa que la vía sea < 10 % de los píxeles). Variantes en el registro `LOSSES`: `focal_dice` (Focal γ + Dice) y `BCE_POS_WEIGHT` (> 1 penaliza más los falsos negativos). **Solo la ablación de `dinov2_partial` puede cambiar la pérdida.**
- **Métricas acumuladas sobre todo el split** (no promedio por parche): `IoU_road`, `mIoU = (IoU_road + IoU_bg) / 2`, `F1`, precisión, recall y **F1 relajado** (ρ = 3 px, Mnih & Hinton 2010).
- **Qué se reporta en test:**
  1. **Oficial** (tabla principal): umbral 0.5, sin TTA. Es lo que se compara entre modelos.
  2. **Umbral calibrado:** el mejor umbral de `VAL_THRESHOLDS` en *validación*, aplicado a test (nunca se elige el umbral mirando test).
  3. **TTA:** promedio de la predicción original + volteo horizontal + volteo vertical, umbral 0.5.

  Los extras 2 y 3 se calculan **igual para los 3 modelos**, así que también son comparables entre ellos, pero van en columnas aparte.

In [ ]:
class SegLoss(nn.Module):
    """bce_dice (base) o focal_dice. Siempre en float32."""

    def __init__(self, cfg, kind=None):
        super().__init__()
        self.kind, self.pos_weight, self.gamma, self.smooth = kind or cfg.LOSS_TYPE, cfg.BCE_POS_WEIGHT, cfg.FOCAL_GAMMA, 1.0

    def dice(self, logits, target):
        p = torch.sigmoid(logits)
        inter = (p * target).sum(dim=(1, 2, 3))
        den = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        return (1.0 - (2.0 * inter + self.smooth) / (den + self.smooth)).mean()

    def forward(self, logits, target):
        logits, target = logits.float(), target.float()
        pw = torch.tensor(self.pos_weight, device=logits.device)
        bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pw, reduction="none")
        if self.kind == "focal_dice":
            pt = torch.exp(-F.binary_cross_entropy_with_logits(logits, target, reduction="none"))
            bce = (1 - pt) ** self.gamma * bce
        return bce.mean() + self.dice(logits, target)


LOSSES = {"bce_dice": lambda cfg: SegLoss(cfg, "bce_dice"), "focal_dice": lambda cfg: SegLoss(cfg, "focal_dice")}


def dilate(x, rho):
    """Dilatación binaria con ventana cuadrada (2*rho+1) vía max-pool."""
    return F.max_pool2d(x, kernel_size=2 * rho + 1, stride=1, padding=rho)


class MetricAccumulator:
    """Acumula TP/FP/FN/TN sobre todo el split y calcula IoU/mIoU/F1/P/R (+ versión relajada)."""

    def __init__(self, rho, thr):
        self.rho, self.thr = rho, thr
        self.c = dict(tp=0, fp=0, fn=0, tn=0, pred_near_gt=0, pred=0, gt_near_pred=0, gt=0)

    @torch.no_grad()
    def update(self, prob, target):
        pred, gt = (prob > self.thr).float(), (target > 0.5).float()
        c = self.c
        c["tp"] += int((pred * gt).sum()); c["fp"] += int((pred * (1 - gt)).sum())
        c["fn"] += int(((1 - pred) * gt).sum()); c["tn"] += int(((1 - pred) * (1 - gt)).sum())
        c["pred"] += int(pred.sum()); c["gt"] += int(gt.sum())
        c["pred_near_gt"] += int((pred * dilate(gt, self.rho)).sum())
        c["gt_near_pred"] += int((gt * dilate(pred, self.rho)).sum())

    def compute(self):
        c, e = self.c, 1e-9
        iou_road = c["tp"] / (c["tp"] + c["fp"] + c["fn"] + e)
        iou_bg = c["tn"] / (c["tn"] + c["fp"] + c["fn"] + e)
        prec, rec = c["tp"] / (c["tp"] + c["fp"] + e), c["tp"] / (c["tp"] + c["fn"] + e)
        rprec, rrec = c["pred_near_gt"] / (c["pred"] + e), c["gt_near_pred"] / (c["gt"] + e)
        return {"iou_road": iou_road, "iou_bg": iou_bg, "miou": (iou_road + iou_bg) / 2,
                "f1": 2 * c["tp"] / (2 * c["tp"] + c["fp"] + c["fn"] + e), "precision": prec, "recall": rec,
                "relaxed_precision": rprec, "relaxed_recall": rrec, "relaxed_f1": 2 * rprec * rrec / (rprec + rrec + e)}


def autocast_ctx(cfg):
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=cfg.AMP and DEVICE.type == "cuda")


@torch.no_grad()
def predict(model, x, cfg, tta=False):
    """Probabilidades (B,1,H,W). tta=True promedia original + volteo horizontal + volteo vertical."""
    with autocast_ctx(cfg):
        logits = model(x)
        if not tta:
            return torch.sigmoid(logits.float()), logits
        p = torch.sigmoid(logits.float())
        p = p + torch.sigmoid(model(x.flip(-1)).float()).flip(-1)
        p = p + torch.sigmoid(model(x.flip(-2)).float()).flip(-2)
    return p / 3.0, logits


@torch.no_grad()
def evaluate(model, loader, cfg, thresholds=(0.5,), tta=False, loss_fn=None, max_batches=None):
    """Devuelve ({umbral: métricas}, loss media). Las probabilidades se calculan una vez por batch."""
    model.eval()
    accs = {t: MetricAccumulator(cfg.RELAX_RHO, t) for t in thresholds}
    total, n = 0.0, 0
    for b, (x, y) in enumerate(loader):
        if max_batches and b >= max_batches:
            break
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        prob, logits = predict(model, x, cfg, tta)
        if loss_fn is not None:
            total += loss_fn(logits, y).item() * x.size(0)
        n += x.size(0)
        for a in accs.values():
            a.update(prob, y)
    return {t: a.compute() for t, a in accs.items()}, total / max(n, 1)


print("Pérdidas disponibles:", list(LOSSES))

## 7. Tu arquitectura en detalle: DINOv2 congelado
### 7.1 Cómo fluye la imagen
```
x (B, 3, 512, 512) ─► reescalado a 518 × 518  (518 = 37 × 14; 512 no es divisible por 14)
 └─ patch_embed: conv 14×14 s14 ─► 37 × 37 = 1369 tokens de 384 dims (+ token CLS)
 └─ 12 bloques Transformer (atención global: cada token mira a los 1369)   [TODOS congelados]
      se guardan las salidas de los bloques DINO_LAYER_IDS = (2, 5, 8, 11) ─► 4 × (B, 384, 37, 37)
 └─ NECK (lo que tú diseñas) — cada mapa 37×37 se lleva a la escala que espera el U-Net:
      bloque 2  ─► bilinear a 128×128 (/4)  ─► 1×1 ─► 64   [+ 3×3 si "bilinear_3x3"]
      bloque 5  ─► bilinear a  64×64  (/8)  ─► 1×1 ─► 128  [+ 3×3 si "bilinear_3x3"]
      bloque 8  ─► bilinear a  32×32  (/16) ─► 1×1 ─► 256
      bloque 11 ─► bilinear a  16×16  (/32) ─► 1×1 ─► 512
                                                       UNetDecoder ─► (B, 1, 512, 512)
```
- **El cuello de botella:** cada token representa un cuadrado de 14 × 14 px. Para llegar a /4 (128 px) hay que **ampliar ×3.5** un mapa de 37 × 37: la interpolación bilineal crea bloques/cuadrícula y bordes borrosos. Una conv 3×3 (aprendible) tras el reescalado puede suavizar ese efecto; un neck con `ConvTranspose2d` puede *aprender* a ampliar.
- **Dónde actúa cada parámetro:** como el backbone está congelado, solo aprenden el neck (~0.37 M) y el decoder (3.1 M). Por eso subir `LR_DECODER` no tiene riesgo de "romper" DINOv2.
- **Ventaja de memoria:** no se guarda el grafo de activaciones del ViT → ~1.3 GB de VRAM en entrenamiento vs ~3 GB de ResNet.

### 7.2 🧪 Zona de código libre (opcional)
Programa tu propio neck aquí y actívalo con `"NECK_TYPE": "custom"`. **Reglas:** firma `(embed_dim, out_chs, cfg)`, `forward(feats, size)` recibe 4 mapas `(B, 384, 37, 37)` y debe devolver 4 mapas con canales `out_chs` y tamaños `size // 4, // 8, // 16, // 32`. No toques `UNetDecoder`.

In [ ]:
class CustomNeck(nn.Module):
    """EJEMPLO: ampliación APRENDIDA. Para /4 y /8 se usan ConvTranspose2d (x2) antes de ajustar el tamaño exacto,
    seguidas de una conv 3x3; /16 y /32 se dejan como la base (bilinear + 1x1)."""
    strides = (4, 8, 16, 32)

    def __init__(self, embed_dim, out_chs, cfg):
        super().__init__()
        self.reduce = nn.ModuleList([ConvBNReLU(embed_dim, o, k=1) for o in out_chs])
        self.up = nn.ModuleList([
            nn.Sequential(nn.ConvTranspose2d(out_chs[0], out_chs[0], 2, stride=2), nn.ConvTranspose2d(out_chs[0], out_chs[0], 2, stride=2)),  # 37 -> 148
            nn.ConvTranspose2d(out_chs[1], out_chs[1], 2, stride=2),                                                                         # 37 -> 74
            nn.Identity(), nn.Identity()])
        self.refine = nn.ModuleList([ConvBNReLU(o, o, 3) if s in (4, 8) else nn.Identity() for o, s in zip(out_chs, self.strides)])

    def forward(self, feats, size):
        H, W = size
        outs = []
        for f, red, up, ref, s in zip(feats, self.reduce, self.up, self.refine, self.strides):
            f = up(red(f))                                                                      # canales -> out_ch, ampliación aprendida
            f = F.interpolate(f, size=(H // s, W // s), mode="bilinear", align_corners=False)   # ajuste al tamaño exacto
            outs.append(ref(f))
        return outs


NECKS["custom"] = CustomNeck
print("Necks disponibles:", list(NECKS))

### 7.3 Recorrido del tensor y parámetros
Construye el modelo de este notebook y muestra la forma del tensor en cada etapa y cuántos parámetros entrena cada componente, primero para la corrida `base` y luego para cada corrida de `RUNS` (así ves, por ejemplo, cuánto cambia el número de parámetros entrenables al tocar `N_UNFREEZE`).

In [ ]:
def make_cfg(overrides):
    """COMMON + MODEL_BASE[CONDITION] + tus cambios. Solo acepta claves de ALLOWED[CONDITION]."""
    bad = sorted(set(overrides) - set(ALLOWED[CONDITION]))
    if bad:
        raise KeyError(f"{bad} no se puede cambiar en {CONDITION}. Permitidos: {ALLOWED[CONDITION]}")
    cfg = SimpleNamespace(**copy.deepcopy({**COMMON, **MODEL_BASE[CONDITION], **overrides}))
    if CONDITION == "resnet50":
        assert cfg.ADAPTER_TYPE in ADAPTERS, f"ADAPTER_TYPE debe ser uno de {list(ADAPTERS)}"
        assert 0 <= cfg.FREEZE_STAGES <= 5
    else:
        assert cfg.NECK_TYPE in NECKS, f"NECK_TYPE debe ser uno de {list(NECKS)}"
        assert 0 <= cfg.N_UNFREEZE <= 12
    assert cfg.LOSS_TYPE in LOSSES, f"LOSS_TYPE debe ser uno de {list(LOSSES)}"
    return cfg


@torch.no_grad()
def describe_model(cfg, verbose=True):
    model = build_model(cfg, CONDITION).to(DEVICE).eval()
    x = torch.zeros(1, 3, cfg.PATCH, cfg.PATCH, device=DEVICE)
    feats = model.encoder(x)
    adapted = model.adapter(feats, tuple(x.shape[-2:]))
    out = model.decoder(adapted, tuple(x.shape[-2:]))
    assert n_params(model.decoder) == EXPECTED_DECODER_PARAMS
    if verbose:
        print("entrada            ", tuple(x.shape))
        for i, (f, a) in enumerate(zip(feats, adapted)):
            print(f"nivel {i}: encoder {str(tuple(f.shape)):<20} -> adaptador/neck {tuple(a.shape)}")
        print("salida (logits)    ", tuple(out.shape))
    rows = [{"componente": n, "total (M)": n_params(m) / 1e6, "entrenables (M)": n_params(m, True) / 1e6}
            for n, m in [("encoder", model.encoder), ("adaptador/neck", model.adapter), ("decoder", model.decoder), ("TOTAL", model)]]
    del model
    torch.cuda.empty_cache()
    return pd.DataFrame(rows).round(3)


print(f"== {CONDITION} · base ==")
display(describe_model(make_cfg({})))
for name, ov in RUNS.items():
    if name != "base":
        print(f"\n== {CONDITION} · {name}: {ov} ==")
        display(describe_model(make_cfg(ov), verbose=False))

## 8. Entrenamiento de las corridas de `RUNS`
Mismo protocolo para todos: AdamW con grupos de LR, *warmup* lineal (5 %) + coseno, AMP FP16, recorte de gradiente 1.0, selección del mejor *checkpoint* por `IoU_road` de validación (umbral 0.5). Por cada corrida se guarda:

- `results/{CONDITION}__{corrida}.json` (configuración completa, cambios respecto a la base, huellas, log por época, test oficial, umbral calibrado, TTA, tiempo y VRAM) y `..._epochs.csv`.
- `figures/` con curvas y ejemplos cualitativos (verde = TP, rojo = FP, azul = FN).

Con `SMOKE_TEST = True` cada corrida hace 1 época corta para comprobar que todo funciona (sus resultados llevan el sufijo `__smoke` y **no** sirven para el informe).

In [ ]:
def make_scaler(cfg):
    on = cfg.AMP and DEVICE.type == "cuda"
    return torch.amp.GradScaler("cuda", enabled=on) if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=on)


def lr_lambda_factory(total_steps, warmup_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return f


def train_one_epoch(model, loader, loss_fn, opt, sched, scaler, trainable, cfg, epoch, max_steps=None):
    model.train()
    total, n, t0 = torch.zeros((), device=DEVICE), 0, time.time()
    pbar = tqdm(loader, desc=f"época {epoch}", leave=False)
    for step, (x, y) in enumerate(pbar):
        if max_steps and step >= max_steps:
            break
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast_ctx(cfg):
            logits = model(x)
        loss = loss_fn(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable, cfg.GRAD_CLIP)
        scaler.step(opt); scaler.update(); sched.step()
        total += loss.detach() * x.size(0); n += x.size(0)
        if step % 25 == 0:
            pbar.set_postfix(loss=f"{total.item() / n:.4f}")
    return total.item() / max(n, 1), time.time() - t0


def plot_curves(log, fname):
    df = pd.DataFrame(log)
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    axes[0].plot(df.epoch, df.train_loss, label="train"); axes[0].plot(df.epoch, df.val_loss, label="val"); axes[0].set_title("Pérdida")
    axes[1].plot(df.epoch, df.val_iou_road, label="IoU road"); axes[1].plot(df.epoch, df.val_f1, label="F1")
    axes[1].plot(df.epoch, df.val_relaxed_f1, label="F1 relajado"); axes[1].set_title("Validación")
    axes[2].plot(df.epoch, df.val_precision, label="precisión"); axes[2].plot(df.epoch, df.val_recall, label="recall"); axes[2].set_title("P / R (val)")
    for ax in axes:
        ax.set_xlabel("época"); ax.grid(alpha=0.3); ax.legend()
    save_fig(fname)


@torch.no_grad()
def qualitative_grid(model, ds, cfg, n, fname):
    model.eval()
    idx = np.linspace(0, len(ds) - 1, num=min(n, len(ds)), dtype=int)
    fig, axes = plt.subplots(len(idx), 4, figsize=(13, 3.2 * len(idx)))
    for r, k in enumerate(idx):
        x, y = ds[int(k)]
        prob, _ = predict(model, x[None].to(DEVICE), cfg)
        pred, gt = (prob[0, 0] > cfg.THRESHOLD).cpu().numpy(), y[0].numpy() > 0.5
        err = np.zeros(gt.shape + (3,), dtype=np.float32)
        err[pred & gt] = (0, 0.8, 0); err[pred & ~gt] = (0.9, 0, 0); err[~pred & gt] = (0, 0.3, 1)
        for ax, im, t, cm in zip(axes[r], [denorm(x), gt, pred, err], ["Imagen", "GT", "Predicción", "TP verde · FP rojo · FN azul"], [None, "gray", "gray", None]):
            ax.imshow(im, cmap=cm); ax.set_title(t, fontsize=8); ax.axis("off")
    save_fig(fname)


def run_experiment(run_name, overrides, smoke=False):
    tag = f"{CONDITION}__{run_name}" + ("__smoke" if smoke else "")
    out_path = f"{RESULTS_DIR}/{tag}.json"
    if SKIP_IF_DONE and os.path.exists(out_path):
        print(f"[{tag}] ya existe {out_path}: se omite (pon SKIP_IF_DONE = False para repetir).")
        return json.load(open(out_path))
    cfg = make_cfg(overrides)
    max_steps, max_eval = (30, 10) if smoke else (None, None)
    if smoke:
        cfg.EPOCHS = 1
    print(f"\n{'=' * 90}\n[{tag}] cambios respecto a la base: {overrides or 'ninguno'}\n{'=' * 90}")
    seed_everything(cfg.SEED)

    train_ds = Patches512(DATA_ROOT, "train", cfg.AUG)
    val_ds, test_ds = Patches512(DATA_ROOT, "val", None), Patches512(DATA_ROOT, "test", None)
    train_loader = make_loader(train_ds, cfg, shuffle=True, drop_last=True)
    val_loader, test_loader = make_loader(val_ds, cfg, False), make_loader(test_ds, cfg, False)
    print(f"parches -> train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

    model = build_model(cfg, CONDITION).to(DEVICE)
    groups = param_groups(model, cfg, CONDITION)
    trainable = [p for g in groups for p in g["params"]]
    counts = {"total": n_params(model), "trainable": n_params(model, True), "encoder_total": n_params(model.encoder),
              "encoder_trainable": n_params(model.encoder, True), "adapter": n_params(model.adapter), "decoder": n_params(model.decoder)}
    print({k: f"{v / 1e6:.2f} M" for k, v in counts.items()})
    print("grupos de LR:", [(g["name"], g["lr"]) for g in groups])

    opt = torch.optim.AdamW(groups)
    steps_per_epoch = min(len(train_loader), max_steps or len(train_loader))
    total_steps = cfg.EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda_factory(total_steps, max(1, int(cfg.WARMUP_FRAC * total_steps))))
    scaler, loss_fn = make_scaler(cfg), LOSSES[cfg.LOSS_TYPE](cfg)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    log, best, ckpt_path, t_start = [], -1.0, f"{CKPT_DIR}/{tag}.pt", time.time()
    for epoch in range(1, cfg.EPOCHS + 1):
        tr_loss, ep_time = train_one_epoch(model, train_loader, loss_fn, opt, sched, scaler, trainable, cfg, epoch, max_steps)
        val_all, val_loss = evaluate(model, val_loader, cfg, (cfg.THRESHOLD,), loss_fn=loss_fn, max_batches=max_eval)
        val_m = val_all[cfg.THRESHOLD]
        log.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": val_loss, "epoch_time_s": ep_time,
                    "lr_head": opt.param_groups[0]["lr"], **{f"val_{k}": v for k, v in val_m.items()}})
        if val_m["iou_road"] > best:
            best = val_m["iou_road"]
            torch.save({"model": model.state_dict(), "epoch": epoch, "val": val_m}, ckpt_path)
        print(f"ep {epoch:02d} | loss {tr_loss:.4f} | val_loss {val_loss:.4f} | val IoU_road {val_m['iou_road']:.4f} | "
              f"F1 {val_m['f1']:.4f} | relF1 {val_m['relaxed_f1']:.4f} | {ep_time:.0f}s")
    train_time = time.time() - t_start
    peak_mb = torch.cuda.max_memory_allocated() / 2 ** 20 if DEVICE.type == "cuda" else None

    ck = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ck["model"])
    sweep, _ = evaluate(model, val_loader, cfg, cfg.VAL_THRESHOLDS, max_batches=max_eval)
    best_thr = max(sweep, key=lambda t: sweep[t]["iou_road"])
    test_all, test_loss = evaluate(model, test_loader, cfg, sorted({cfg.THRESHOLD, best_thr}), loss_fn=loss_fn, max_batches=max_eval)
    test_tta, _ = evaluate(model, test_loader, cfg, (cfg.THRESHOLD,), tta=True, max_batches=max_eval)
    test_m = test_all[cfg.THRESHOLD]
    print(f"\nTEST oficial (ckpt época {ck['epoch']}, umbral {cfg.THRESHOLD}): " + " | ".join(f"{k}={v:.4f}" for k, v in test_m.items()))
    print(f"TEST umbral calibrado en val ({best_thr}): IoU_road={test_all[best_thr]['iou_road']:.4f} | "
          f"TEST con TTA: IoU_road={test_tta[cfg.THRESHOLD]['iou_road']:.4f}")

    plot_curves(log, f"{tag}_curves")
    qualitative_grid(model, test_ds, cfg, 4, f"{tag}_qualitative")
    out = {"condition": CONDITION, "run": run_name, "smoke": smoke, "overrides": overrides, "config": cfg,
           "locked_fingerprint": LOCKED_FINGERPRINT, "dataset_fingerprint": DATASET_FP,
           "n_patches": {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
           "params": counts, "lr_groups": [(g["name"], g["lr"]) for g in groups],
           "epochs": log, "best_epoch": ck["epoch"], "best_val": ck["val"],
           "test": test_m, "test_loss": test_loss,
           "val_threshold_sweep": {str(t): m for t, m in sweep.items()}, "best_val_threshold": best_thr,
           "test_at_best_val_threshold": test_all[best_thr], "test_tta": test_tta[cfg.THRESHOLD],
           "train_time_s": train_time, "peak_vram_mb": peak_mb,
           "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu", "torch": torch.__version__}
    with open(out_path, "w") as f:
        json.dump(to_jsonable(out), f, indent=2)
    pd.DataFrame(log).to_csv(f"{RESULTS_DIR}/{tag}_epochs.csv", index=False)
    print("Escrito:", out_path)
    del model, opt, scaler
    torch.cuda.empty_cache()
    return to_jsonable(out)

In [ ]:
ALL_RESULTS = {}
for name, ov in RUNS.items():
    ALL_RESULTS[name] = run_experiment(name, ov, smoke=SMOKE_TEST)

## 9. Comparación base vs. ablación
Tabla con todas las corridas de este modelo (las de esta sesión y, si subes JSON de sesiones anteriores como *input*, también esas). `Δ` = diferencia contra la fila `base` (en puntos de IoU/F1 × 100). Una ablación solo es válida si **las dos huellas coinciden** con la base.

In [ ]:
def load_results(condition):
    paths = glob.glob(f"{RESULTS_DIR}/{condition}__*.json") + glob.glob(f"/kaggle/input/**/{condition}__*.json", recursive=True)
    rows, seen = [], set()
    for p in sorted(paths):
        r = json.load(open(p))
        key = (r["run"], r["smoke"])
        if key in seen:
            continue
        seen.add(key)
        rows.append({"run": r["run"] + (" (smoke)" if r["smoke"] else ""), "cambios": json.dumps(r["overrides"]),
                     "IoU_road": r["test"]["iou_road"], "mIoU": r["test"]["miou"], "F1": r["test"]["f1"],
                     "relaxed_F1": r["test"]["relaxed_f1"], "precision": r["test"]["precision"], "recall": r["test"]["recall"],
                     "thr_val": r["best_val_threshold"], "IoU_road@thr_val": r["test_at_best_val_threshold"]["iou_road"],
                     "IoU_road_TTA": r["test_tta"]["iou_road"], "best_epoch": r["best_epoch"],
                     "entrenables_M": r["params"]["trainable"] / 1e6, "tiempo_min": r["train_time_s"] / 60,
                     "VRAM_MB": r["peak_vram_mb"], "huella_cfg": r["locked_fingerprint"], "huella_datos": r["dataset_fingerprint"]})
    df = pd.DataFrame(rows)
    if df.empty:
        print("Sin resultados todavía.")
        return df
    base = df[df.run == "base"]
    if len(base):
        for c in ["IoU_road", "F1", "relaxed_F1"]:
            df[f"Δ {c} (pts)"] = 100 * (df[c] - base.iloc[0][c])
    if df.huella_cfg.nunique() > 1 or df.huella_datos.nunique() > 1:
        print("AVISO: hay corridas con huellas distintas: no son comparables entre sí.")
    return df


COMPARISON = load_results(CONDITION)
if len(COMPARISON):
    display(COMPARISON.round(4))
    COMPARISON.to_csv(f"{RESULTS_DIR}/{CONDITION}__comparison.csv", index=False)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for name, r in ALL_RESULTS.items():
        ep = pd.DataFrame(r["epochs"])
        ax[0].plot(ep.epoch, ep.val_iou_road, label=name); ax[1].plot(ep.epoch, ep.train_loss, label=name)
    ax[0].set_title("val IoU_road por época"); ax[1].set_title("train loss por época")
    for a in ax:
        a.set_xlabel("época"); a.grid(alpha=0.3); a.legend()
    save_fig(f"{CONDITION}__comparison_curves")

## 10. Qué enviar al grupo
Se crea `outputs_{CONDITION}.zip` con `results/` y `figures/` (sin los checkpoints, que pesan ~100 MB cada uno). En Kaggle aparece en la pestaña *Output*; en Colab se descarga con el panel de archivos. **Envía el zip** (o al menos los `.json`) al grupo.

In [ ]:
_tmp = f"/tmp/outputs_{CONDITION}"
shutil.rmtree(_tmp, ignore_errors=True)
shutil.copytree(RESULTS_DIR, f"{_tmp}/results")
shutil.copytree(FIG_DIR, f"{_tmp}/figures")
zip_path = shutil.make_archive(f"{WORK}/outputs_{CONDITION}", "zip", root_dir=_tmp)
print("Listo para enviar:", zip_path)